# 01. Getting Started with VAFT

This 90-minute worksheet introduces a VEST tokamak shot, its diagnostics, and VAFT's public analysis and plotting interfaces.

It is a **hands-on worksheet, not a demonstration**. Several cells contain a `BLANK` that you have to replace with your own answer, and the notebook will stop at the first one you have not filled in. Pressing *Run All* on a fresh copy is expected to fail; that is the point.

The reproducible core uses the VEST sample packaged with VAFT, shot 39915. Two later sections use larger reference datasets that ship with the repository checkout, and an optional lab branch reads the public HSDS database when it has been configured.

## Session Overview

By the end of this session you will be able to:

- describe a tokamak shot as a time-resolved experiment with many diagnostics;
- explain the roles of VAFT, OMAS, IMAS-style interface data structures (IDSs), and VEST data;
- list which diagnostics VEST has, and which of them a given shot actually contains;
- identify what each major VAFT namespace is for;
- inspect engineering geometry and diagnostic geometry, and relate a signal to where it was measured;
- locate representative diagnostic channels in an ODS and plot them; and
- read a signal's time range, sampling rate, units, and channel identity before drawing physical conclusions.

Suggested timing: 15 minutes for setup and context, 20 minutes for the inventory and API tour, 20 minutes for geometry, 25 minutes for the diagnostic signals, and 10 minutes for the integrated and independent exercises.

### Before you start

Your environment should already be set up. If it is not, follow
[`install/README.md`](../install/README.md) and then run:

```bash
python install/check_vaft_environment.py
```

Environment setup is not part of this lesson. Budget 15-20 minutes for it beforehand, and ask for help rather than spending the session on it.

### How the exercises work

Each exercise is a Markdown cell describing the task, followed by a code cell that is mostly written for you. You fill in the `BLANK` placeholders:

```python
time = ods[BLANK]        # TODO: the plasma-current time path
require(3, time=time)
```

`BLANK` refuses to be used as a value, so an unanswered exercise stops with an instruction rather than a confusing error further down. A `check_values(...)` or `require(...)` call then confirms your answer before the rest of the cell runs.

Once every exercise is answered correctly, the notebook runs from top to bottom without manual intervention.

## Physical Context

A **shot** is one time-resolved tokamak discharge. Its data describe how programmed coil currents, magnetic fields, plasma current, and diagnostic signals evolve from prefill and breakdown, through the plasma phase, to shutdown.

A diagnostic signal is not a result by itself. Before it means anything you need four things: a time reference, the units it is stored in, which channel produced it, and where in the machine that channel sits. This session is built around that discipline.

VAFT organizes these data as OMAS objects using the IMAS vocabulary. Practically, an **IDS** is a named top-level data family such as `magnetics`, `pf_active`, `equilibrium`, or `spectrometer_uv`. Treat the structure as a map: find the relevant IDS, inspect a documented path, then call a public VAFT function.

Two distinctions matter throughout:

- **What VEST has** (its full diagnostic inventory) versus **what this shot contains** (the IDSs actually present in one ODS). They are not the same, and you will compare them directly in a moment.
- **Engineering geometry** (vessel, limiter, coils, passive structure) versus **diagnostic geometry** (where each probe, loop, or chord sits). Together they answer: where did this signal come from?

## Load / Prepare Data

The setup cell chooses the execution mode, creates an ignored directory for the figures your run produces, puts the exercise helpers on the import path, and loads the packaged sample. It does not contact HSDS in offline mode.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import vaft

MODE = os.environ.get("VAFT_TUTORIAL_MODE", "offline").strip().lower()
if MODE not in {"offline", "lab"}:
    raise ValueError("VAFT_TUTORIAL_MODE must be 'offline' or 'lab'")


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "tutorial").is_dir():
            return candidate
    raise RuntimeError(
        "Run this notebook from the VAFT checkout, or set VAFT_TUTORIAL_OUTPUT_DIR."
    )


SHOT = 39915
REPOSITORY_ROOT = find_repository_root(Path.cwd())
TUTORIAL_DIR = REPOSITORY_ROOT / "tutorial"
if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))

from exercise_support import BLANK, check_values, confirm, require  # noqa: E402

OUTPUT_DIR = Path(
    os.environ.get("VAFT_TUTORIAL_OUTPUT_DIR", TUTORIAL_DIR / "outputs" / "01")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ods = vaft.omas.sample_ods()

print(f"Mode: {MODE}; source: VAFT packaged sample; shot: {SHOT}")
print(f"Figures for this run: {OUTPUT_DIR}")


### What VEST has, and what this shot contains

VAFT ships a machine-level diagnostic registry describing every VEST diagnostic, whether it is routinely operated, and whether VAFT can map it into an IDS yet. That is a property of the *machine*.

A single shot contains only a subset of it. Read this table before going looking for a signal: it tells you which diagnostics exist, which are routinely operated, and which VAFT can map into an IDS at all. Exercise 1 then compares it against what shot 39915 actually holds.

`available_plot_names` is a third thing again: what VAFT can *plot* from this particular ODS. You will use it repeatedly.

In [ ]:
from vaft.machine_mapping.registry import load_diagnostic_registry

registry = load_diagnostic_registry()
print(f"VEST diagnostic registry: {len(registry)} entries\n")
print(f"{'diagnostic':<34}{'IDS':<22}{'availability':<14}mapping")
print("-" * 84)
for record in sorted(registry.values(), key=lambda item: str(item.get("name", ""))):
    print(
        f"{str(record.get('name', ''))[:33]:<34}"
        f"{str(record.get('ids', '-'))[:21]:<22}"
        f"{str(record.get('availability', '-'))[:13]:<14}"
        f"{record.get('mapping_status', '-')}"
    )

available_plot_names = sorted(row["name"] for row in vaft.omas.available_plots(ods))
print(f"\nVAFT can draw {len(available_plot_names)} plot recipes from this ODS.")
print("Examples:", ", ".join(available_plot_names[:8]))


### Exercise 1 -- Find your way around the ODS

An ODS behaves like a dictionary whose keys are dotted IMAS paths. Before plotting anything, establish where the magnetic diagnostics live.

1. List the IDS roots this shot contains, using `sorted(ods.keys())`.
2. Decide which root holds the magnetic diagnostics -- plasma current, flux loops, and magnetic probes.
3. Every IDS stores its own time base at `<ids>.time`. Write that path for the IDS you chose.

The registry table above tells you which IDS each diagnostic maps to. The cell also prints which registered diagnostics are *missing* from this shot -- worth a glance.

Fill in both blanks below.

In [ ]:
# TODO: replace both blanks.
magnetics_ids = BLANK        # the IDS root holding the magnetic diagnostics
magnetics_time_path = BLANK  # that IDS's time base, as a dotted path

require(1, magnetics_ids=magnetics_ids, magnetics_time_path=magnetics_time_path)

ids_roots = sorted(ods.keys())
check_values(
    1,
    ids_is_present=(magnetics_ids in ids_roots, True),
    time_path_is_consistent=(magnetics_time_path, f"{magnetics_ids}.time"),
)

time_s = np.asarray(ods[magnetics_time_path])
print(f"This shot contains {len(ids_roots)} IDS roots:")
for name in ids_roots:
    print(f"- {name}")
print(
    f"\n{magnetics_time_path}: {time_s.size} samples "
    f"from {time_s.min():.4f} to {time_s.max():.4f} s"
)

# `not_developed` is the registry's placeholder for a diagnostic with no IDS yet.
mapped_ids = {
    str(record.get("ids"))
    for record in registry.values()
    if record.get("ids") and str(record.get("ids")) != "not_developed"
}
print(f"\nRegistered VEST diagnostics present in shot {SHOT}: {sorted(mapped_ids & set(ids_roots))}")
print(f"Registered but absent from this shot:            {sorted(mapped_ids - set(ids_roots))}")


### Optional lab branch: read-only HSDS access

Set `VAFT_TUTORIAL_MODE=lab` only after configuring credentials with `hsconfigure`. This branch reads the `magnetics` IDS and nothing else. When the read succeeds, the plasma-current plot later on overlays the live trace on the packaged sample; everything else stays reproducible offline.

Expected connection or credential failures are reported as a skip. Genuine API errors still stop the notebook, because those are bugs rather than missing access.

In [ ]:
lab_time_s = None
lab_ip_a = None

if MODE == "lab":
    try:
        with vaft.database.open(SHOT, source="public", paths="magnetics") as remote_ods:
            lab_time_s = np.array(remote_ods["magnetics.time"], copy=True)
            lab_ip_a = np.array(remote_ods["magnetics.ip.0.data"], copy=True)
        print(f"HSDS read succeeded: {lab_time_s.size} magnetic time samples.")
    except OSError as error:
        print("Lab extension skipped: HSDS read-only access is unavailable.")
        print(f"Reason: {type(error).__name__}: {error}")
else:
    print("Lab extension skipped in offline mode; the packaged sample remains active.")


## Guided Analysis

### A tour of the VAFT namespaces

VAFT is organised by *what you are doing*, not by diagnostic. One representative call from each major namespace is enough to build the mental model; this is orientation, not an API reference.

| Namespace | Use it to |
| --- | --- |
| `vaft.data` | find packaged datasets and read file formats |
| `vaft.omas` | load, inspect, derive from, and plot an ODS |
| `vaft.machine_mapping` | turn raw VEST signals and machine geometry into IDSs |
| `vaft.process` | process signals and compute physics quantities |
| `vaft.plot` | render a prepared view model (called for you by `vaft.omas.plot_*`) |
| `vaft.database` | load shots from the remote VEST database |

The division between `vaft.plot` and `vaft.omas.plot_*` is worth noting: `vaft.plot.<name>` draws a prepared view model and refuses an ODS, while `vaft.omas.plot_<name>` extracts that view model from an ODS and then draws it. In a notebook you almost always want the `vaft.omas` adapter.

In [ ]:
# vaft.data -- where packaged datasets live
print("packaged samples:", vaft.data.available_samples())

# vaft.omas -- shot-level introspection
print("shot number:      ", vaft.omas.find_shotnumber(ods))
print("peak plasma current:", f"{vaft.omas.find_max_ip(ods):.0f} A")

# vaft.machine_mapping -- machine description, independent of any shot
print("registry entries: ", len(registry))

# vaft.process -- signal processing on plain arrays
ip_a = np.asarray(ods["magnetics.ip.0.data"])
smoothed_ip_a = vaft.process.smooth(ip_a, 51)
print("smoothing changed the peak by:", f"{abs(ip_a).max() - abs(smoothed_ip_a).max():.0f} A")

# vaft.plot -- the renderer layer, which deliberately rejects an ODS
try:
    vaft.plot.magnetics_time_ip(ods)
except TypeError as error:
    print("vaft.plot refuses an ODS, as designed:")
    print(f"  {error}")


### Geometry first

Geometry comes before signal plotting on purpose. A trace with no location attached is hard to interpret and easy to over-read.

```text
machine geometry + diagnostic geometry
                  |
                  v
      where the measured signal comes from
```

Start with the engineering side: the vacuum vessel and limiter outline, the poloidal-field coils, and the passive conducting structure. `machine_geometry_poloidal` composes all of them into one poloidal cross-section.

In [ ]:
fig_machine, ax_machine = vaft.omas.plot_machine_geometry_poloidal(ods, show=False)
ax_machine.set_title(f"VEST shot {SHOT}: engineering geometry")
fig_machine.savefig(
    OUTPUT_DIR / "session01_machine_geometry.png", dpi=180, bbox_inches="tight"
)
display(fig_machine)
plt.close(fig_machine)

wall_r = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.r"])
wall_z = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.z"])
print(f"limiter outline: {wall_r.size} points")
print(f"  R from {wall_r.min():.3f} to {wall_r.max():.3f} m")
print(f"  Z from {wall_z.min():.3f} to {wall_z.max():.3f} m")
print(f"PF coils: {len(ods['pf_active.coil'])}")
print(f"passive structure loops: {len(ods['pf_passive.loop'])}")


### Exercise 2 -- Diagnostic geometry

Now the diagnostic side. The magnetic diagnostics are a set of flux loops and poloidal-field probes at fixed positions around the poloidal cross-section.

1. Find the recipe in `available_plot_names` that draws flux-loop and B-field-probe positions in the poloidal plane. Its name follows the `<ids>_geometry_<view>` convention.
2. Each flux loop stores its position under `magnetics.flux_loop.<index>.position.0.r` and `...z`. Read the first loop's `R` and `Z`.

Compare the printed position with where that point falls on the plot.

In [ ]:
# TODO: replace all three blanks.
geometry_plot = BLANK  # a name from available_plot_names
flux_loop_r = BLANK    # R of magnetics.flux_loop.0, in metres
flux_loop_z = BLANK    # Z of magnetics.flux_loop.0, in metres

require(2, geometry_plot=geometry_plot, flux_loop_r=flux_loop_r, flux_loop_z=flux_loop_z)
check_values(
    2,
    recipe_is_available=(geometry_plot in available_plot_names, True),
    r_matches_the_ods=(
        round(float(flux_loop_r), 6),
        round(float(ods["magnetics.flux_loop.0.position.0.r"]), 6),
    ),
    z_matches_the_ods=(
        round(float(flux_loop_z), 6),
        round(float(ods["magnetics.flux_loop.0.position.0.z"]), 6),
    ),
)

fig_geometry, ax_geometry = getattr(vaft.omas, f"plot_{geometry_plot}")(ods, show=False)
ax_geometry.set_title(f"VEST shot {SHOT}: diagnostic geometry")
fig_geometry.savefig(
    OUTPUT_DIR / "session01_diagnostic_geometry.png", dpi=180, bbox_inches="tight"
)
display(fig_geometry)
plt.close(fig_geometry)

print(f"flux loops:            {len(ods['magnetics.flux_loop'])}")
print(f"poloidal field probes: {len(ods['magnetics.b_field_pol_probe'])}")
print(f"{ods['magnetics.flux_loop.0.name']}: R = {flux_loop_r:.3f} m, Z = {flux_loop_z:.3f} m")


### Representative diagnostic signals

Work through one diagnostic at a time, always in the same order:

```text
find the diagnostic -> identify the channel -> inspect time / data / units
                    -> plot it -> read the behaviour
```

Start with the single most important trace on any tokamak: the plasma current, measured by a Rogowski coil.

### Exercise 3 -- Locate the plasma current

The plasma current lives in the `magnetics` IDS. VAFT stores it as an array of measurements, so it is indexed: `magnetics.ip.<index>.data`, with `<index>` starting at 0.

1. Write the path to the plasma-current time base. You worked this out in Exercise 1.
2. Write the path to the plasma-current data for measurement 0.
3. Run the cell. It reports the sample count, time span, and peak value before plotting.

In [ ]:
# TODO: replace both blanks with dotted ODS paths.
ip_time_path = BLANK  # the magnetics time base
ip_data_path = BLANK  # plasma-current data for measurement 0

require(3, ip_time_path=ip_time_path, ip_data_path=ip_data_path)

ip_time_s = np.asarray(ods[ip_time_path])
ip_a = np.asarray(ods[ip_data_path])
check_values(3, arrays_align=(ip_time_s.size == ip_a.size, True))

print(f"samples:   {ip_a.size}")
print(f"time span: {ip_time_s.min():.4f} to {ip_time_s.max():.4f} s")
print(f"peak |Ip|: {np.abs(ip_a).max():.0f} A")


In [ ]:
fig_ip, ax_ip = vaft.omas.plot_magnetics_time_ip(ods, show=False)
ax_ip.set_title(f"VEST shot {SHOT}: plasma current")
if lab_time_s is not None and lab_ip_a is not None:
    ax_ip.plot(
        lab_time_s,
        lab_ip_a,
        linestyle="--",
        linewidth=1.5,
        label=f"{SHOT} live HSDS",
    )
    ax_ip.legend()
fig_ip.savefig(OUTPUT_DIR / "session01_plasma_current.png", dpi=180, bbox_inches="tight")
display(fig_ip)
plt.close(fig_ip)


### Exercise 4 -- Equilibrium magnetics

Flux loops measure poloidal flux; B-field probes measure the local poloidal field. Both are multi-channel systems, and this shot has many channels. Expose the whole inventory, then plot a small representative subset -- plotting all 76 probes would tell you nothing.

1. Choose three flux-loop channel indices to plot. Any three valid indices will do.
2. Choose two B-field-probe channel indices.

The `channels=` option is accepted by every multi-channel `vaft.omas.plot_*` adapter.

In [ ]:
print("flux loop channels:")
for index in range(len(ods["magnetics.flux_loop"])):
    print(f"  {index:2d}  {ods[f'magnetics.flux_loop.{index}.name']}")

probe_count = len(ods["magnetics.b_field_pol_probe"])
print(f"\nB-field probe channels: {probe_count} (showing the first 5)")
for index in range(min(5, probe_count)):
    print(f"  {index:2d}  {ods[f'magnetics.b_field_pol_probe.{index}.name']}")

# TODO: replace both blanks with lists of channel indices.
flux_loop_channels = BLANK  # three flux-loop indices, e.g. [0, 1, 2]
probe_channels = BLANK      # two B-field-probe indices

require(4, flux_loop_channels=flux_loop_channels, probe_channels=probe_channels)
check_values(
    4,
    three_flux_loops=(len(flux_loop_channels), 3),
    two_probes=(len(probe_channels), 2),
    flux_loops_in_range=(
        all(0 <= index < len(ods["magnetics.flux_loop"]) for index in flux_loop_channels),
        True,
    ),
    probes_in_range=(all(0 <= index < probe_count for index in probe_channels), True),
)


In [ ]:
fig_flux, ax_flux = vaft.omas.plot_magnetics_time_flux_loop_flux(
    ods, channels=flux_loop_channels, show=False
)
fig_flux.savefig(OUTPUT_DIR / "session01_flux_loops.png", dpi=180, bbox_inches="tight")
display(fig_flux)
plt.close(fig_flux)

fig_probe, ax_probe = vaft.omas.plot_magnetics_time_b_field_pol_probe_field(
    ods, channels=probe_channels, show=False
)
fig_probe.savefig(OUTPUT_DIR / "session01_b_field_probes.png", dpi=180, bbox_inches="tight")
display(fig_probe)
plt.close(fig_probe)

fig_diamagnetic, ax_diamagnetic = vaft.omas.plot_magnetics_time_diamagnetic_flux(
    ods, show=False
)
fig_diamagnetic.savefig(
    OUTPUT_DIR / "session01_diamagnetic_flux.png", dpi=180, bbox_inches="tight"
)
display(fig_diamagnetic)
plt.close(fig_diamagnetic)


### Machine and edge signals

Three more diagnostic classes, plotted without exercises so you can concentrate on reading them:

- **PF coil currents** -- the programmed waveforms that shape and position the plasma.
- **Toroidal field** -- the TF coil current and the resulting toroidal field.
- **H-alpha filterscope** -- visible line emission, a standard proxy for recycling and edge neutrals.
- **Barometry** -- neutral pressure in the vessel, which sets the prefill condition.

Watch how the timing of these relates to the plasma-current trace you plotted above.

In [ ]:
for name, filename in (
    ("pf_active_time_current", "session01_pf_active_current.png"),
    ("tf_time_b_field_tor", "session01_tf_field.png"),
    ("spectrometer_uv_time_intensity", "session01_halpha.png"),
    ("barometry_time_pressure", "session01_barometry.png"),
):
    figure, _ = getattr(vaft.omas, f"plot_{name}")(ods, show=False)
    figure.savefig(OUTPUT_DIR / filename, dpi=180, bbox_inches="tight")
    display(figure)
    plt.close(figure)

print("H-alpha / filterscope channels:")
for index in range(len(ods["spectrometer_uv.channel"])):
    print(f"  {index}  {ods[f'spectrometer_uv.channel.{index}.name']}")
print("barometry gauges:")
for index in range(len(ods["barometry.gauge"])):
    print(f"  {index}  {ods[f'barometry.gauge.{index}.name']}")


### Profile diagnostics, from a second reference dataset

The packaged shot 39915 carries magnetics, coils, spectroscopy, and an equilibrium, but no kinetic profile diagnostics. Those come from a second reference dataset in the repository: shot 48224 at 300 ms, which holds Thomson scattering, charge exchange, and the fitted core profiles.

This file ships with the **repository checkout**, not with the wheel, so the cell below skips cleanly if you installed VAFT from PyPI. Nothing later depends on it.

Thomson scattering measures electron temperature and density at a set of fixed positions along a laser path; charge exchange recombination spectroscopy gives ion temperature and toroidal rotation.

In [ ]:
kinetic_path = vaft.data.data_path("kineticEfit/ods_48224_300ms.json")

if not kinetic_path.is_file():
    print("Skipped: this reference dataset requires the repository checkout.")
    print(f"Expected at: {kinetic_path}")
else:
    kinetic_ods = vaft.omas.load(kinetic_path)
    print("IDS roots:", sorted(kinetic_ods.keys()))
    print(f"Thomson channels:        {len(kinetic_ods['thomson_scattering.channel'])}")
    print(f"charge-exchange channels: {len(kinetic_ods['charge_exchange.channel'])}")

    for name, filename in (
        ("thomson_scattering_geometry_poloidal", "session01_thomson_geometry.png"),
        ("thomson_scattering_profile_electron_temperature", "session01_thomson_te.png"),
        ("thomson_scattering_profile_electron_density", "session01_thomson_ne.png"),
        ("charge_exchange_profile_ion_temperature", "session01_cx_ti.png"),
    ):
        figure, _ = getattr(vaft.omas, f"plot_{name}")(kinetic_ods, show=False)
        figure.savefig(OUTPUT_DIR / filename, dpi=180, bbox_inches="tight")
        display(figure)
        plt.close(figure)


### Interferometry, mapped from a raw file

Everything so far read an ODS that someone else had already built. This cell shows the step before that: `vaft.machine_mapping` turning a raw diagnostic file into a populated IDS.

The 94 GHz horizontal interferometer measures the line-integrated electron density along several chords. The raw file is a downsampled export from shot 47230 that ships with the repository checkout.

In [ ]:
from omas import ODS

interferometer_file = vaft.data.data_path("legacy/47230_056789_LID_1_100.mat")

if not interferometer_file.is_file():
    print("Skipped: this reference dataset requires the repository checkout.")
    print(f"Expected at: {interferometer_file}")
else:
    interferometer_ods = ODS()
    vaft.machine_mapping.interferometer_94ghz(
        interferometer_ods,
        47230,
        mat_file=interferometer_file,
        compute_line_average=True,
    )
    channels = interferometer_ods["interferometer.channel"]
    print(f"mapped {len(channels)} interferometer chords from the raw file")
    for index in range(len(channels)):
        print(f"  {index}  {interferometer_ods[f'interferometer.channel.{index}.identifier']}")

    figure, _ = vaft.omas.plot_interferometer_time_n_e_line(interferometer_ods, show=False)
    figure.savefig(
        OUTPUT_DIR / "session01_interferometer.png", dpi=180, bbox_inches="tight"
    )
    display(figure)
    plt.close(figure)


### Soft X-ray emission, from a raw digitizer export

One more diagnostic class, and one more reminder that a dataset is chosen to suit a question. Soft X-ray arrays view the plasma through many collimated lines of sight, so their emission profile carries information about the shape and position of the hot core and about impurity radiation.

Neither shot 39915 nor shot 48224 has soft X-ray data, so this section uses a third dataset: shot 45531, stored as a raw digitizer CSV in the repository checkout. VEST records this diagnostic on two DAQs; the cell below maps one of them, the 40-channel vertical array, which takes about ten seconds.

Note what the mapping call needs that the earlier ones did not: a `daq_label` identifying which digitizer to read. Diagnostics are not uniform, and their mapping functions reflect that.

In [ ]:
from vaft.machine_mapping.soft_x_rays import soft_x_rays

SXR_SHOT = 45531
SXR_DAQ = 17592
legacy_root = vaft.data.data_path("legacy")
sxr_file = legacy_root / f"digitizer_{SXR_DAQ}_{SXR_SHOT}.csv"

if not sxr_file.is_file():
    print("Skipped: this reference dataset requires the repository checkout.")
    print(f"Expected at: {sxr_file}")
else:
    print(f"Mapping the vertical soft X-ray array for shot {SXR_SHOT} ...")
    sxr_ods = ODS()
    soft_x_rays(sxr_ods, SXR_SHOT, SXR_DAQ, data_root=legacy_root)

    sxr_channels = sxr_ods["soft_x_rays.channel"]
    brightness = np.asarray(sxr_ods["soft_x_rays.channel.0.brightness.data"])
    sxr_time_s = np.asarray(sxr_ods["soft_x_rays.channel.0.brightness.time"])
    print(f"channels:  {len(sxr_channels)}")
    print(f"channel 0: {sxr_ods['soft_x_rays.channel.0.name']}")
    print(f"           {brightness.size} samples "
          f"from {sxr_time_s.min():.4f} to {sxr_time_s.max():.4f} s")

    sxr_plot_names = sorted(row["name"] for row in vaft.omas.available_plots(sxr_ods))
    print(f"recipes for this ODS: {sxr_plot_names}")

    for name, filename in (
        ("soft_x_rays_geometry_lines_of_sight", "session01_sxr_lines_of_sight.png"),
        ("soft_x_rays_overview", "session01_sxr_overview.png"),
    ):
        figure, _ = getattr(vaft.omas, f"plot_{name}")(sxr_ods, show=False)
        figure.savefig(OUTPUT_DIR / filename, dpi=180, bbox_inches="tight")
        display(figure)
        plt.close(figure)


## Interpretation Checkpoints

Before drawing any conclusion from a trace, answer these five questions about it. They are the difference between reading data and guessing at it.

1. What is the time range, and what is the approximate sampling rate?
2. What unit is the quantity stored in?
3. Which diagnostic and which channel produced it?
4. When does the signal rise or change, and what is its characteristic value?
5. Where is that channel in the machine, relative to the plasma?

Discuss these too:

- Which interval of the plasma-current trace is the sustained plasma phase, and what evidence supports that?
- Which magnetic or spectroscopic signal changes near the current rise? Does that establish causation, or only correlation?
- What extra information would you need before making a stronger physical claim?

Exercise 5 makes the first three questions concrete.

### Exercise 5 -- Read the signal properly

Using the plasma-current arrays from Exercise 3:

1. Compute the sampling rate in Hz from the time base. The samples are evenly spaced, so the mean interval is enough.
2. Read the stored unit for the plasma current. IMAS keeps it at `magnetics.ip.0.data` metadata -- but in this ODS the reliable source is the IDS documentation, so state it yourself: the plasma current is stored in amperes, `"A"`.
3. Read the channel identity: the name of the first flux loop is `ods["magnetics.flux_loop.0.name"]`; the plasma current comes from the Rogowski coil.

Fill in the sampling rate and the unit.

In [ ]:
# TODO: replace both blanks.
sampling_rate_hz = BLANK  # from ip_time_s: 1 / mean spacing, in Hz
ip_unit = BLANK           # the unit the plasma current is stored in

require(5, sampling_rate_hz=sampling_rate_hz, ip_unit=ip_unit)

expected_rate_hz = 1.0 / float(np.mean(np.diff(ip_time_s)))
check_values(
    5,
    sampling_rate_is_right=(round(float(sampling_rate_hz), -2), round(expected_rate_hz, -2)),
    unit_is_right=(str(ip_unit), "A"),
)

duration_s = float(ip_time_s.max() - ip_time_s.min())
peak_index = int(np.argmax(np.abs(ip_a)))
print(f"time range:    {ip_time_s.min():.4f} to {ip_time_s.max():.4f} s ({duration_s * 1e3:.1f} ms)")
print(f"sampling rate: {expected_rate_hz / 1e3:.1f} kHz over {ip_a.size} samples")
print(f"unit:          {ip_unit}")
print(f"peak |Ip|:     {np.abs(ip_a)[peak_index] / 1e3:.1f} kA at t = {ip_time_s[peak_index]:.4f} s")
print("source:        Rogowski coil, via magnetics.ip.0")


## Integrated Analysis

In real analysis the diagnostic is often chosen at runtime rather than written into the code. The pattern is: pick a recipe name from `available_plot_names`, resolve the matching `vaft.omas.plot_*` function with `getattr`, and call it.

This is exactly what you did by hand in Exercise 2, now as a deliberate technique.

### Exercise 6 -- Resolve a recipe at runtime

1. Choose any recipe from `available_plot_names` that you have **not** already plotted in this notebook.
2. Set `selected_plot` to its name.
3. Run the cell. It resolves the function, plots it, and records your choice.

If your choice needs a `channels=` argument it will still work; the adapters default to a sensible selection.

In [ ]:
already_plotted = {
    "machine_geometry_poloidal",
    "magnetics_geometry_poloidal",
    "magnetics_time_ip",
    "magnetics_time_flux_loop_flux",
    "magnetics_time_b_field_pol_probe_field",
    "magnetics_time_diamagnetic_flux",
    "pf_active_time_current",
    "tf_time_b_field_tor",
    "spectrometer_uv_time_intensity",
    "barometry_time_pressure",
}

# TODO: replace the blank with a recipe name you have not plotted yet.
selected_plot = BLANK

require(6, selected_plot=selected_plot)
check_values(
    6,
    recipe_is_available=(selected_plot in available_plot_names, True),
    recipe_is_new=(selected_plot in already_plotted, False),
)

selected_function = getattr(vaft.omas, f"plot_{selected_plot}")
fig_selected, axes_selected = selected_function(ods, show=False)
fig_selected.savefig(
    OUTPUT_DIR / f"session01_{selected_plot}.png", dpi=180, bbox_inches="tight"
)
display(fig_selected)
plt.close(fig_selected)

print(f"resolved vaft.omas.plot_{selected_plot} at runtime")


## Independent Exercise

Choose one more diagnostic recipe, plot it, and interpret it in writing.

1. Set `exercise_plot` to a recipe from `available_plot_names` that is different from every plot made so far.
2. Run the cell.
3. Add a new Markdown cell below and write three sentences:
   - which diagnostic and which time interval you examined;
   - one feature visible in the trace or image; and
   - one additional measurement or piece of metadata you would need to interpret that feature physically.

The five interpretation checkpoints above are your checklist. Good starting choices are `magnetics_overview`, `equilibrium_field_psi`, `equilibrium_profile_q`, or `summary_time_voltage_consumption`.

In [ ]:
# TODO: replace the blank with your chosen recipe.
exercise_plot = BLANK

require(7, exercise_plot=exercise_plot)
check_values(
    7,
    recipe_is_available=(exercise_plot in available_plot_names, True),
    recipe_is_new=(exercise_plot in already_plotted | {selected_plot}, False),
)

exercise_function = getattr(vaft.omas, f"plot_{exercise_plot}")
fig_exercise, axes_exercise = exercise_function(ods, show=False)
fig_exercise.savefig(
    OUTPUT_DIR / f"session01_exercise_{exercise_plot}.png", dpi=180, bbox_inches="tight"
)
display(fig_exercise)
plt.close(fig_exercise)

print(f"You plotted {exercise_plot}. Now write your three sentences in a new Markdown cell below.")


## Takeaways and Next Steps

You loaded a packaged VEST ODS, compared the machine's diagnostic inventory against what one shot actually contains, inspected engineering and diagnostic geometry before plotting anything, and read representative signals from the magnetic, coil, spectroscopic, edge-pressure, profile, interferometry, and soft X-ray diagnostics. You also saw the step below all of that: `vaft.machine_mapping` building an IDS from a raw file.

Along the way you used four different datasets. That is normal: no single shot carries every diagnostic, and choosing the dataset that answers your question is part of the work.

The habit worth keeping is the checkpoint list -- time range, units, channel identity, characteristic behaviour, and position -- applied before any physical claim.

Spectral and mode analysis of the fluctuation and soft X-ray signals is deliberately left for Session 04.

The same offline-first pattern continues through the course: start from a reproducible input, use a public VAFT API, and add a gated lab or solver extension only when the environment supports it.

Session 02 connects these diagnostic traces to discharge operation and vacuum fields. For broader plotting examples, see
[Plotting Sample Data with VAFT Plot Module](../notebooks/plotting_sample_using_vaft_plot_module.ipynb).

In [ ]:
print(
    f"SESSION_01_OFFLINE_READY: shot={SHOT}; ids={len(ids_roots)}; "
    f"plots={len(available_plot_names)}; output_dir={OUTPUT_DIR}"
)
